In [1]:
import spacy
import medspacy
from medspacy.visualization import visualize_ent, visualize_dep

In [2]:
with open("./texto_historia_v2.txt") as f:
    text = f.read()

In [3]:
nlp = medspacy.load(medspacy_enable=["medspacy_pyrush"])

### Extraccion de objetivos:

Usando texto_historia_v2, extraeremos los siguientes conceptos:
* **Diagnósticos**
* **Medicamentos**

Además, mostraremos algunos ejemplos de cómo añadir un atributo personalizado de **spaCy** a una regla de objetivo (`TargetRule`) para agregar un código de diagnóstico **CIE-10** como atributo de una entidad.

Primero, importarmos `TargetMatcher` y `TargetRule`. Instanciaremos un `TargetRule` y lo añadiremos a nuestro pipeline.

In [4]:
target_matcher = nlp.add_pipe("medspacy_target_matcher")

La clase `TargetRule` define reglas para la extracción de entidades. `TargetRule` acepta los siguientes argumentos:

* **literal**: Una frase exacta para coincidir en el texto.
* **category**: La clase semántica de la entidad; corresponderá a `ent.label_` en un objeto `Doc`.
* **pattern**: Un patrón opcional para coincidir en lugar del `literal`. Puede ser una lista de diccionarios que definen atributos de tokens o una cadena de expresión regular.
* **on_match**: Una función de *callback* opcional. Consulta: https://spacy.io/usage/rule-based-matching#on_match
* **metadata**: Un diccionario opcional de metadatos.
* **attributes**: Un diccionario opcional de atributos personalizados para establecer en la entidad resultante.

In [5]:
from medspacy.ner import TargetRule

In [6]:
# Definimos las reglas de extracción (TargetRules) adaptadas al texto en español
# literal: frase exacta en el texto
# category: etiqueta semántica de la entidad
target_rules = [
    TargetRule(literal="dolor abdominal", category="PROBLEMA"),
    TargetRule("accidente cerebrovascular", "PROBLEMA"),
    TargetRule("hemicolectomía", "TRATAMIENTO"),
    TargetRule("Hidroclorotiazida", "TRATAMIENTO"),
    TargetRule("cáncer de colon", "PROBLEMA"),
    TargetRule("metástasis", "PROBLEMA"),
]

In [7]:
target_matcher.add(target_rules)

In [8]:
doc = nlp(text)

2026-03-17 11:09:01.733 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 0 'Fecha' marked as sentence start (span begin)
2026-03-17 11:09:01.734 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 23 '

' marked as sentence start (span end whitespace)
2026-03-17 11:09:01.735 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] GAP DETECTED: tokens 23-23 (idx 69-69) between spans 69-71
2026-03-17 11:09:01.736 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 23 '

' marked as sentence start (whitespace in gap between spans)
2026-03-17 11:09:01.737 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 24 'Fecha' marked as sentence start (span begin)
2026-03-17 11:09:01.737 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] To

In [9]:
print(doc.ents)

(Hidroclorotiazida, Dolor abdominal, accidente cerebrovascular, dolor abdominal, metástasis, Cáncer de colon, hemicolectomía, dolor abdominal)


In [10]:
for ent in doc.ents:
    print(ent, ent.label_, ent._.target_rule, sep="  |  ")
    print()

Hidroclorotiazida  |  TRATAMIENTO  |  TargetRule(literal="Hidroclorotiazida", category="TRATAMIENTO", pattern=None, attributes=None, on_match=None)

Dolor abdominal  |  PROBLEMA  |  TargetRule(literal="dolor abdominal", category="PROBLEMA", pattern=None, attributes=None, on_match=None)

accidente cerebrovascular  |  PROBLEMA  |  TargetRule(literal="accidente cerebrovascular", category="PROBLEMA", pattern=None, attributes=None, on_match=None)

dolor abdominal  |  PROBLEMA  |  TargetRule(literal="dolor abdominal", category="PROBLEMA", pattern=None, attributes=None, on_match=None)

metástasis  |  PROBLEMA  |  TargetRule(literal="metástasis", category="PROBLEMA", pattern=None, attributes=None, on_match=None)

Cáncer de colon  |  PROBLEMA  |  TargetRule(literal="cáncer de colon", category="PROBLEMA", pattern=None, attributes=None, on_match=None)

hemicolectomía  |  TRATAMIENTO  |  TargetRule(literal="hemicolectomía", category="TRATAMIENTO", pattern=None, attributes=None, on_match=None)

dol

Agregamos reglas para identificar radioterapia y dm tipo 2

In [11]:
pattern_rules = [
    
    
    TargetRule("radioterapia", "PROBLEMA",
              pattern=[{"LOWER": {"IN": ["rtx", "radioterapia"]}}]
              ),
    
    
    TargetRule("diabetes", "PROBLEMA",
              pattern=r"(dm|diabetes mellitus) tipo (i|ii|1|2|uno|dos)"),
    
                ]

### Explicación del Patrón de Coincidencia (spaCy)

El código `pattern=[{"LOWER": {"IN": ["rtx", "radioterapia"]}}]` es una regla lógica que instruye al extractor a buscar un solo token que cumpla con las siguientes condiciones:

1.  **`{...}`**: Cada par de llaves representa un **token** (una palabra, número o signo de puntuación) individual en el texto.
2.  **`"LOWER"`**: Indica que la coincidencia no debe distinguir entre mayúsculas y minúsculas. El motor convertirá el texto del documento a minúsculas antes de compararlo.
3.  **`{"IN": ["rtx", "radioterapia"]}`**: Es un operador lógico de pertenencia. Significa que el patrón será exitoso si el texto del token (en minúsculas) es exactamente **"rtx"** O **"radioterapia"**.

Este patrón permite encontrar menciones de "Radioterapia" en el texto, incluso si están escritas como "RTX", "rtx", "Radioterapia" o "RADIOTERAPIA", tratándolas a todas como la misma entidad.

In [12]:
target_matcher.add(pattern_rules)

In [13]:
doc = nlp(text)

2026-03-17 11:09:10.648 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 0 'Fecha' marked as sentence start (span begin)
2026-03-17 11:09:10.649 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 23 '

' marked as sentence start (span end whitespace)
2026-03-17 11:09:10.649 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] GAP DETECTED: tokens 23-23 (idx 69-69) between spans 69-71
2026-03-17 11:09:10.650 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 23 '

' marked as sentence start (whitespace in gap between spans)
2026-03-17 11:09:10.650 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 24 'Fecha' marked as sentence start (span begin)
2026-03-17 11:09:10.651 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] To

In [14]:
print(doc.ents)

(Hidroclorotiazida, Dolor abdominal, DM tipo 2, accidente cerebrovascular, dolor abdominal, metástasis, Cáncer de colon, hemicolectomía, RTX, Diabetes Mellitus Tipo II, DM Tipo 2, dolor abdominal)


### Añadiendo atributos personalizados

Una de las funcionalidades más potentes de **spaCy** es la capacidad de añadir [atributos personalizados](https://spacy.io/usage/processing-pipelines#custom-components-attributes) a los objetos de spaCy (`Doc`, `Span` o `Token`). Se accede a estos atributos personalizados a través de `obj._....`. **medSpaCy** añade varios de estos atributos en otros componentes como `context` o `sectionizer`.

A veces es útil incluir un atributo personalizado como parte de la regla de coincidencia de objetivos (*target matching*), en lugar de tener que construir un componente separado para añadirlo. La clase `TargetRule` puede incluir un valor para estos atributos en el argumento `attributes`.

Por ejemplo, supongamos que queremos mapear ciertas entidades a códigos de diagnóstico [CIE-10](https://es.wikipedia.org/wiki/CIE-10). Una forma de hacerlo es incluir los códigos de diagnóstico para los conceptos en nuestra base de conocimientos. Por ejemplo, "Diabetes Tipo II" puede mapearse a **"E11.9"**. Podemos añadir esto a las entidades registrando primero la extensión para la clase `Span`:

In [15]:
from spacy.tokens import Span

In [16]:
Span.set_extension("icd10", default="")

Ahora podemos incluir los valores de los códigos **CIE-10** en las reglas de objetivos. Mapearemos "Diabetes Mellitus Tipo II" a **"E11.9"** e "Hipertensión" a **"I10"**:

In [17]:
# Definimos una segunda lista de reglas con atributos personalizados (CIE-10)
target_rules2 = [
    TargetRule(
        literal="Diabetes Mellitus Tipo II", 
        category="PROBLEMA", 
        pattern=[
            {"LOWER": {"IN":["dm", "diabetes"]}},
            {"LOWER": "mellitus", "OP": "?"}, # El "OP": "?" hace que sea opcional
            {"LOWER": "tipo"},
            {"LOWER": {"IN": ["2", "ii", "dos"]}}
        ],
        attributes={"icd10": "E11.9"}
    ),
    TargetRule(
        literal="Hipertensión", 
        category="PROBLEMA",
        pattern=[{"LOWER": {"IN": ["hta", "hipertension", "hipertensión"]}}],
        attributes={"icd10": "I10"}
    ),
]

In [18]:
target_matcher.add(target_rules2)

In [19]:

doc = nlp(text)

2026-03-17 11:09:32.717 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 0 'Fecha' marked as sentence start (span begin)
2026-03-17 11:09:32.718 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 23 '

' marked as sentence start (span end whitespace)
2026-03-17 11:09:32.719 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] GAP DETECTED: tokens 23-23 (idx 69-69) between spans 69-71
2026-03-17 11:09:32.720 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 23 '

' marked as sentence start (whitespace in gap between spans)
2026-03-17 11:09:32.721 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 24 'Fecha' marked as sentence start (span begin)
2026-03-17 11:09:32.722 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] To

In [20]:
for ent in doc.ents:
    if ent._.icd10 != "":
        print(ent, ent._.icd10)

DM tipo 2 E11.9
Diabetes Mellitus Tipo II E11.9
Hipertensión I10
DM Tipo 2 E11.9
HTA I10


### Contexto (ConText)

El texto clínico a menudo contiene menciones de conceptos que el paciente no experimentó realmente. Por ejemplo:

* "No hay evidencia de neumonía" (Negación)
* "Madre con cáncer de mama" (Sujeto/Experimentador)
* "Paciente se presenta para descartar COVID-19" (Incertidumbre)

En todos estos casos, necesitamos utilizar las pistas contextuales alrededor de la entidad para afirmar atributos como la negación, el experimentador y la incertidumbre.

Un método para esto es el **algoritmo ConText**. ConText vincula entidades objetivo (como problemas) con modificadores semánticos como los mostrados anteriormente. La implementación de ConText en medSpaCy se encuentra en `medspacy.context`.

El componente ConText se explica con más detalle en los notebooks de la carpeta `context/`.

**NOTA:** ConText requiere que se establezcan límites de oraciones (*sentence boundaries*) o ventanas de proximidad.

In [21]:
from medspacy.context import ConTextRule

In [22]:
# Añadimos una regla específica para "Madre" en español
context_rules = [
    ConTextRule(literal="madre", category="FAMILIAR", direction="FORWARD")
]

In [23]:
nlp.add_pipe("medspacy_context", last=True)
nlp.pipe_names

['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context']

In [24]:
context = nlp.get_pipe("medspacy_context")
context.add(context_rules)

In [25]:
nlp.pipe_names

['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context']

In [26]:
# Procesamos la oración en español usando el modelo nlp
doc = nlp("Madre con accidente cerebrovascular a la edad de 82 años.")

2026-03-17 11:09:45.731 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 0 'Madre' marked as sentence start (span begin)
2026-03-17 11:09:45.734 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] Token/tag mapping: [(Madre, True), (con, False), (accidente, False), (cerebrovascular, False), (a, False), (la, False), (edad, False), (de, False), (82, False), (años, False), (., False)]


In [27]:
visualize_ent(doc)

In [28]:
visualize_dep(doc)

In [29]:
# Definimos reglas de contexto para detectar antecedentes históricos
context_rules_2 = [
    ConTextRule(
        literal="diagnosticado en <AÑO>", 
        category="HISTORICO", 
        direction="BACKWARD", # Mira "hacia atrás" en el texto (a la izquierda)
        pattern=[
            {"LOWER": {"IN": ["diagnosticado", "diagnosticada", "dx"]}},
            {"LOWER": "en"},
            {"LOWER": {"REGEX": r"^[\d]{4}$"}} # Busca un número de 4 dígitos (el año)
        ]
    )
]

In [30]:
nlp.pipe_names

['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context']

In [31]:
context.add(context_rules_2)

In [32]:
# Procesamos la oración en español para probar el modificador histórico
short_doc = nlp("Cáncer de colon diagnosticado en 2012")

2026-03-17 11:10:01.919 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] [doc 0] Token 0 'Cáncer' marked as sentence start (span begin)
2026-03-17 11:10:01.920 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] Token/tag mapping: [(Cáncer, True), (de, False), (colon, False), (diagnosticado, False), (en, False), (2012, False)]


In [33]:
visualize_ent(short_doc)

In [34]:
visualize_dep(short_doc)